# Week 11 — Block 2: Guided Demo (Consuming APIs & Data Endpoints)

**DATS 6401 · Visualization of Complex Data**

The two worked examples from the module, live (~35 min):

- **Example A** (~15 min): a public no-key API (Open-Meteo) → JSON → pandas → chart. *Requires internet.*
- **Example B** (~12 min): the **course endpoint** → status codes, error handling, caching. *Requires the instructor's `course_endpoint.py` running on port 8001 (see that file's header).*
- Wrap-up (~8 min): the same calls inside a Streamlit client → open `client_starter.py`.


## Example A — public API → pandas → chart

Anatomy of the call: a **URL**, **query parameters**, a **timeout**, and a **status check**. Narrate each.

In [ ]:
import requests
import pandas as pd

r = requests.get(
    "https://api.open-meteo.com/v1/forecast",          # the endpoint
    params={"latitude": 38.9, "longitude": -77.0,       # query parameters (DC)
            "hourly": "temperature_2m"},
    timeout=10,                                         # NEVER call without a timeout
)
print("Status:", r.status_code)
r.raise_for_status()                                    # raises on 4xx/5xx

payload = r.json()                                      # JSON -> Python dicts/lists
print("Top-level keys:", list(payload.keys()))

In [ ]:
import pandas as pd
# (payload comes from the previous cell)
# JSON -> tidy DataFrame -> chart
h = payload["hourly"]
df = pd.DataFrame({"time": pd.to_datetime(h["time"]), "temp_c": h["temperature_2m"]})
df.plot(x="time", y="temp_c", figsize=(9, 3), title="DC temperature forecast (Open-Meteo)");

**Narrate:** the entire pattern is four steps — *request → check status → parse JSON → DataFrame*. Everything else this week is variations.

## Example B — the course endpoint: status codes & failure

Start the server first (separate terminal): `uvicorn course_endpoint:app --port 8001`

In [ ]:
BASE = "http://127.0.0.1:8001"

r = requests.get(f"{BASE}/data", params={"species": "setosa"}, timeout=5)
print("Status:", r.status_code)
print("Count:", r.json()["count"])
pd.DataFrame(r.json()["records"]).head(3)

In [ ]:
# Deliberately break it: the endpoint 404s on unknown species
r = requests.get(f"{BASE}/data", params={"species": "dragon"}, timeout=5)
print("Status:", r.status_code)        # 404
print("Body:", r.json())

# The robust pattern: raise_for_status inside try/except
try:
    r.raise_for_status()
except requests.HTTPError as e:
    print("Caught:", e)

**Narrate:** a non-200 is not an exception until *you* make it one — `raise_for_status()` + `try/except` is the discipline. A client that crashes on a 404 is a broken client.

## Wrap-up: the same call in Streamlit

Open `client_starter.py`, run it (`streamlit run client_starter.py`), and point out: the `@st.cache_data(ttl=300)` wrapper around the fetch, and the `st.error` path when the server is down. **Kill the server live and show the app degrade gracefully.** That's the behavior the homework grades.